# 🛠️ setup_colab.ipynb — Daily GitHub loop (Colab)


# 🔁 Daily Repo Loop for **Ego4d-LiteSTA** (Colab-first)

This block sets up SSH to GitHub (**Deploy Key stored in Drive**), pulls your repo, runs code, logs outputs, and pushes logs/artifacts back to GitHub.
- One-time: add your **Deploy Key** public key to your GitHub repo Deploy Keys with **write access**.
- Edit the variables (**GITHUB_USER**, **REPO_NAME**) below once; they persist per notebook.


In [ ]:
# ⛳ Mount Drive and set variables
from google.colab import drive
drive.mount('/content/drive')

GITHUB_USER = "your-gh-username"          # <<< EDIT ME
REPO_NAME   = "Ego4d-LiteSTA"             # <<< default as requested
REPO_SSH    = f"git@github.com:{GITHUB_USER}/{REPO_NAME}.git"
WORKDIR     = f"/content/{REPO_NAME}"
SSH_DIR     = "/content/drive/MyDrive/.ssh"

print("Repo:", REPO_SSH)
print("Workdir:", WORKDIR)

In [ ]:
# 🔐 SSH + git config + clone/pull
# 1) Git identity (optional but recommended)
!git config --global user.name "Your Name"
!git config --global user.email "your@email.com"

# 2) SSH agent and key from Drive (key filename: colab_egolite)
%%bash -s "$SSH_DIR"
SSH_DIR="$1"
mkdir -p "$SSH_DIR"
chmod 700 "$SSH_DIR"
eval "$(ssh-agent -s)"
ssh-add "$SSH_DIR/colab_egolite" >/dev/null 2>&1 || true
mkdir -p ~/.ssh && chmod 700 ~/.ssh
cat <<EOF > ~/.ssh/config
Host github.com
  HostName github.com
  User git
  IdentityFile $SSH_DIR/colab_egolite
  IdentitiesOnly yes
StrictHostKeyChecking accept-new
EOF
chmod 600 ~/.ssh/config
ssh-keyscan -t rsa,ed25519 github.com >> ~/.ssh/known_hosts 2>/dev/null || true
chmod 644 ~/.ssh/known_hosts

# 3) Clone or pull
import os
if not os.path.exists(WORKDIR):
    !git clone "$REPO_SSH" "$WORKDIR"
else:
    %cd "$WORKDIR"
    !git pull --rebase
%cd "$WORKDIR"
!mkdir -p runs logs outputs notebooks
print("Repo ready at", WORKDIR)

In [ ]:
# 🧾 Run ID + logger helpers (use these in your train/eval cells)
import os, json, traceback, datetime, shutil, glob

RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_DIR = "logs"; RUN_DIR = f"runs/{RUN_ID}"; OUT_DIR = "outputs"
os.makedirs(LOG_DIR, exist_ok=True); os.makedirs(RUN_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

LOG_PATH = f"{LOG_DIR}/{RUN_ID}.log"
ERR_FILE = "COLAB_ERRORS.md"

def log_write(msg):
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(msg + "\n")
    print(msg)

def record_error(title, err_text):
    with open(ERR_FILE, "a", encoding="utf-8") as f:
        f.write(f"\n## {RUN_ID} — {title}\n\n```\n{err_text}\n```\n")

log_write(f"New run: {RUN_ID}")

➡️ After your training/eval cells, run the next cell to push logs/outputs back:

In [ ]:
# 🚀 Commit & push logs/artifacts
import json, glob, datetime
%cd "$WORKDIR"

# Optional: include metrics summary in the commit message if exists
metrics_candidates = sorted(glob.glob("runs/*/metrics.json"))
metrics_note = ""
if metrics_candidates:
    try:
        with open(metrics_candidates[-1]) as f:
            m = json.load(f)
        metrics_note = f" | sel_acc={m.get('selector_acc')} verb_acc={m.get('verb_acc')} ttc_mae={m.get('ttc_mae')}"
    except Exception as e:
        print("Could not read metrics.json:", e)

!git add logs/* runs/* outputs/* COLAB_ERRORS.md || true
!git status --porcelain

COMMIT_MSG = f"colab: logs & outputs {datetime.datetime.now().isoformat(timespec='seconds')}{metrics_note}"
!git commit -m "$COMMIT_MSG" || echo "Nothing to commit."
!git push
print("Pushed to GitHub.")